In [ ]:
!nvidia-smi

Thu Apr 30 00:27:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q sentence-transformers faiss-cpu datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.2 MB/s eta 0:00:00


Load Wikipedia and pick a subset

In [ ]:
from datasets import load_dataset

print("Loading Wikipedia 2023 English dump (this is a streaming load)...")
wiki_full = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True,
)

# Grab the first 50,000 articles
print("Selecting 50,000 articles...")
articles = []
for i, art in enumerate(wiki_full):
    if i >= 50000:
        break
    articles.append({"title": art["title"], "text": art["text"]})
    if (i + 1) % 10000 == 0:
        print(f"  loaded {i+1}/50000")

print(f"Loaded {len(articles)} Wikipedia articles")
print(f"\nExample article title: {articles[0]['title']}")
print(f"Example text (first 200 chars): {articles[0]['text'][:200]}...")

Loading Wikipedia 2023 English dump (this is a streaming load)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Selecting 50,000 articles...
  loaded 10000/50000
  loaded 20000/50000
  loaded 30000/50000
  loaded 40000/50000
  loaded 50000/50000
Loaded 50000 Wikipedia articles

Example article title: Anarchism
Example text (first 200 chars): Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typi...


Chunk articles into passages

In [ ]:
def chunk_text(text, chunk_size=100, overlap=20):
    """Chunk text into overlapping windows of words."""
    words = text.split()
    chunks = []
    if len(words) <= chunk_size:
        if len(words) >= 30:  # skip very short
            chunks.append(" ".join(words))
        return chunks
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        if len(chunk) >= 30:
            chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
    return chunks


print("Chunking articles into passages...")
passages = []
for art in articles:
    title = art["title"]
    chunks = chunk_text(art["text"], chunk_size=100, overlap=20)
    for chunk in chunks:
        passages.append({"title": title, "text": chunk})

print(f"Created {len(passages)} passages from {len(articles)} articles")
print(f"Average passages per article: {len(passages) / len(articles):.1f}")
print(f"\nExample passage:\n  Title: {passages[0]['title']}\n  Text: {passages[0]['text'][:200]}...")

Chunking articles into passages...
Created 525731 passages from 50000 articles
Average passages per article: 10.5

Example passage:
  Title: Anarchism
  Text: Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typi...


Embed all passages

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading sentence-transformer embedding model...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

print(f"Encoding {len(passages)} passages (takes ~15-20 min)...")
passage_texts = [p["text"] for p in passages]
passage_embeddings = embedder.encode(
    passage_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)

print(f"Encoded {passage_embeddings.shape[0]} passages → shape {passage_embeddings.shape}")

Using device: cuda
Loading sentence-transformer embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 525731 passages (takes ~15-20 min)...


Batches:   0%|          | 0/4108 [00:00<?, ?it/s]

Encoded 525731 passages → shape (525731, 384)


 Build FAISS index

In [ ]:
import faiss

print("Building FAISS index...")
dim = passage_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(passage_embeddings)
print(f"FAISS index built with {index.ntotal} vectors of dim {dim}")

Building FAISS index...
FAISS index built with 525731 vectors of dim 384


Saving everything to Drive

In [ ]:
import json
import os

CORPUS_DIR = "/content/drive/MyDrive/wiki_corpus_50k"
os.makedirs(CORPUS_DIR, exist_ok=True)

print(f"Saving FAISS index...")
faiss.write_index(index, f"{CORPUS_DIR}/wiki.faiss")

print(f"Saving passages metadata...")
with open(f"{CORPUS_DIR}/passages.json", "w") as f:
    json.dump(passages, f)

print(f"Saving embeddings (numpy)...")
np.save(f"{CORPUS_DIR}/embeddings.npy", passage_embeddings)

print(f"\n All saved to {CORPUS_DIR}")
!du -sh /content/drive/MyDrive/wiki_corpus_50k/
!ls -la /content/drive/MyDrive/wiki_corpus_50k/

Saving FAISS index...
Saving passages metadata...
Saving embeddings (numpy)...

 All saved to /content/drive/MyDrive/wiki_corpus_50k
1.9G	/content/drive/MyDrive/wiki_corpus_50k/
total 1916803
-rw------- 1 root root 807522944 Apr 30 00:57 embeddings.npy
-rw------- 1 root root 347759166 Apr 30 00:57 passages.json
-rw------- 1 root root 807522861 Apr 30 00:57 wiki.faiss


Test the new retriever on TriviaQA samples

In [ ]:
def retrieve_passages(question, k=5):
    """Retrieve top-k passages for a question."""
    q_emb = embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        p = passages[int(idx)]
        results.append({
            "title": p["title"],
            "text": p["text"],
            "score": float(score),
        })
    return results

# Testing on the same questions that failed before
test_questions = [
    "Who wrote the play Hamlet?",
    "What was first worn by British soldiers in India in 1845?",
    "Which US gangster was released from Alcatraz prison in November 1939?",
    "What is the capital of Australia?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"QUESTION: {q}")
    print('='*70)
    results = retrieve_passages(q, k=3)
    for i, p in enumerate(results, 1):
        print(f"\n[{i}] (score={p['score']:.3f}) {p['title']}")
        print(f"    {p['text'][:200]}...")


QUESTION: Who wrote the play Hamlet?

[1] (score=0.594) Words, Words, Words
    act of Hamlet after being inspired by Swift's suggestion to poison Dr. Rosenbaum. Productions Words, Words, Words premiered in January 1987, in the Manhattan Punch Line Theatre in New York City. It st...

[2] (score=0.564) Mucedorus
    1973. Tucker Brooke, C.F., ed., The Shakespeare Apocrypha, Oxford, the Clarendon Press, 1908. archive.org Google Books External links The Comedy of Mucedorus, ed. Warnke and Proescholdt (Halle, 1878) ...

[3] (score=0.561) Michael Culver
    and James Ward. Not in the Book by Arthur Watkyn. Directed by Raymond Westwell. The Vanity Case by Jack Popplewell. Directed by Raymond Westwell. Charley's Aunt by Brandon Thomas. Directed by Raymond ...

QUESTION: What was first worn by British soldiers in India in 1845?

[1] (score=0.628) Red coat (military uniform)
    serving soldiers' opinion showed little support for the idea and it was shelved. Colonial forces throughout the Empi

In [ ]:
retriever_code = '''"""
retriever_v2.py — Scaled retriever using 50k Wikipedia articles
Author: DATA 612 Group Project
"""

import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

CORPUS_DIR = "/content/drive/MyDrive/wiki_corpus_50k"
_device = "cuda" if torch.cuda.is_available() else "cpu"
_embedder = None
_index = None
_passages = None


def load_retriever():
    global _embedder, _index, _passages
    print("Loading scaled retriever (50k Wikipedia articles)...")
    _embedder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device=_device
    )
    _index = faiss.read_index(f"{CORPUS_DIR}/wiki.faiss")
    with open(f"{CORPUS_DIR}/passages.json") as f:
        _passages = json.load(f)
    print(f"Retriever ready. Index has {_index.ntotal} passages.")


def retrieve_passages(question, k=5):
    if _index is None:
        load_retriever()
    q_emb = _embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    scores, indices = _index.search(q_emb, k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        p = _passages[int(idx)]
        results.append({
            "title": p["title"],
            "text": p["text"],
            "score": float(score),
        })
    return results


if __name__ == "__main__":
    load_retriever()
    for r in retrieve_passages("Who wrote Hamlet?", k=3):
        print(f"[{r[\\"score\\"]:.3f}] {r[\\"title\\"]}: {r[\\"text\\"][:150]}")
'''

with open('/content/drive/MyDrive/retriever_v2.py', 'w') as f:
    f.write(retriever_code)

print("Saved retriever_v2.py to Drive")

Saved retriever_v2.py to Drive


**Run RAG on full 500 questions**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("HF token loaded")

HF token loaded


In [ ]:
!pip install -q transformers==4.44.2 accelerate==0.33.0 datasets==2.21.0 sentence-transformers faiss-cpu
!pip install -q "bitsandbytes>=0.46.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.

Load the scaled retriever from Drive

In [ ]:
import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

CORPUS_DIR = "/content/drive/MyDrive/wiki_corpus_50k"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading sentence-transformer embedder...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

print("Loading FAISS index from Drive...")
index = faiss.read_index(f"{CORPUS_DIR}/wiki.faiss")
print(f"  Index has {index.ntotal} vectors")

print("Loading passages from Drive...")
with open(f"{CORPUS_DIR}/passages.json") as f:
    passages = json.load(f)
print(f"  Loaded {len(passages)} passages")

def retrieve_passages(question, k=5):
    q_emb = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    return [{"title": passages[int(i)]["title"], "text": passages[int(i)]["text"], "score": float(s)}
            for i, s in zip(indices[0], scores[0])]

print("Retriever ready")

# Quick sanity check
for r in retrieve_passages("Who wrote Hamlet?", k=2):
    print(f"  [{r['score']:.3f}] {r['title']}")

Loading sentence-transformer embedder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading FAISS index from Drive...
  Index has 525731 vectors
Loading passages from Drive...
  Loaded 525731 passages
Retriever ready
  [0.558] The Gathering (Carmody novel)
  [0.524] Michael Culver


Load Mistral-7B in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Mistral tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Mistral model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Mistral loaded")

Loading Mistral tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral loaded


RAG generation function

In [ ]:
def generate_answer(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def build_rag_prompt(question, retrieved):
    passages_text = "\n\n".join([f"[Passage {i+1}] {p['text']}" for i, p in enumerate(retrieved)])
    return (
        f"[INST] You are given several passages from Wikipedia that may contain "
        f"the answer to a question. Use the passages to answer the question with "
        f"a short, concise answer. If the passages do not contain the answer, "
        f"answer based on general knowledge.\n\n"
        f"Passages:\n{passages_text}\n\n"
        f"Question: {question} [/INST] Answer:"
    )


def rag_answer(question, k=5):
    retrieved = retrieve_passages(question, k=k)
    prompt = build_rag_prompt(question, retrieved)
    return generate_answer(prompt), retrieved


# Quick test
ans, ret = rag_answer("Who wrote the play Hamlet?", k=5)
print("Question: Who wrote the play Hamlet?")
print(f"Top retrieved: {ret[0]['title']}")
print(f"Answer: {ans}")

Question: Who wrote the play Hamlet?
Top retrieved: Words, Words, Words
Answer: William Shakespeare wrote the play Hamlet. However, the passage does not provide information about who wrote the specific production of Hamlet mentioned in Passage 5. That production was directed by Andrei Tarkovsky in 1976.


Run RAG on full 500 TriviaQA questions

In [ ]:
from datasets import load_dataset
import random
import json
from tqdm import tqdm

print("Loading TriviaQA validation set...")
triviaqa_val = load_dataset("trivia_qa", "rc.nocontext", split="validation")
random.seed(42)
eval_indices = random.sample(range(len(triviaqa_val)), 500)
eval_set = triviaqa_val.select(eval_indices)
print(f"Eval slice: {len(eval_set)} questions")

print("\nRunning RAG on 500 TriviaQA questions...")
rag_results = []
for ex in tqdm(eval_set, desc="RAG"):
    question = ex["question"]
    gold = ex["answer"]["value"]
    aliases = ex["answer"]["aliases"]
    answer, retrieved = rag_answer(question, k=5)
    rag_results.append({
        "question": question,
        "gold": gold,
        "aliases": aliases,
        "prediction": answer,
        "retrieved_titles": [r["title"] for r in retrieved],
        "top_passage_score": retrieved[0]["score"] if retrieved else 0.0,
    })

with open("/content/drive/MyDrive/rag_full_results.json", "w") as f:
    json.dump(rag_results, f, indent=2)

print(f"\n Saved {len(rag_results)} RAG predictions to Drive")
print(f"\nFirst 3 predictions:")
for r in rag_results[:3]:
    print(f"\nQ: {r['question']}")
    print(f"Gold: {r['gold']}")
    print(f"Pred: {r['prediction'][:120]}")

Loading TriviaQA validation set...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

Eval slice: 500 questions

Running RAG on 500 TriviaQA questions...


RAG: 100%|██████████| 500/500 [37:23<00:00,  4.49s/it]


 Saved 500 RAG predictions to Drive

First 3 predictions:

Q: What was first worn by British soldiers in India in 1845?
Gold: Khaki
Pred: Khaki uniforms were first introduced for Indian and colonial warfare by the British Army in the mid-19th century, around

Q: Which US gangster was released from Alcatraz prison in November 1939?
Gold: Al Capone
Pred: Al Capone was released from Alcatraz prison in November 1939.

Q: Of which tribe of Red Indians was Geronimo a chief
Gold: Apache
Pred: Geronimo was a chief of the Chiricahua Apache tribe. He is not mentioned in the given passages.


Quick EM/F1 on the results

In [ ]:
import re, string

def normalize(s):
    if not s: return ""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(c for c in s if c not in string.punctuation)
    return " ".join(s.split())

def em_strict(pred, gold, aliases):
    pn = normalize(pred)
    return int(any(normalize(c) == pn for c in [gold] + list(aliases) if c))

def em_substring(pred, gold, aliases):
    pn = normalize(pred)
    return int(any(normalize(c) in pn for c in [gold] + list(aliases) if normalize(c)))

n = len(rag_results)
strict = sum(em_strict(r["prediction"], r["gold"], r["aliases"]) for r in rag_results) / n
substr = sum(em_substring(r["prediction"], r["gold"], r["aliases"]) for r in rag_results) / n
print(f"\n{'='*50}")
print(f"RAG (no fine-tune) on {n} TriviaQA questions:")
print(f"  Strict EM:    {strict:.3f}")
print(f"  Substring EM: {substr:.3f}")
print(f"{'='*50}")


RAG (no fine-tune) on 500 TriviaQA questions:
  Strict EM:    0.014
  Substring EM: 0.588


**RAG + LoRA**

Attach the trained LoRA adapter to the loaded Mistral

In [ ]:
from peft import PeftModel

ADAPTER_PATH = "/content/drive/MyDrive/lora_checkpoints/mistral-lora-triviaqa-final/checkpoint-625"

print(f"Loading LoRA adapter from {ADAPTER_PATH}...")
model_lora = PeftModel.from_pretrained(model, ADAPTER_PATH)
model_lora.eval()
print("LoRA adapter attached to Mistral")

# Verify it loaded correctly
trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_lora.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.4f}%)")

Loading LoRA adapter from /content/drive/MyDrive/lora_checkpoints/mistral-lora-triviaqa-final/checkpoint-625...
LoRA adapter attached to Mistral
Trainable params: 0 (0.0000%)


sanity test before the big run

In [ ]:
def generate_answer_lora(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model_lora.device)
    with torch.no_grad():
        output = model_lora.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def rag_lora_answer(question, k=5):
    retrieved = retrieve_passages(question, k=k)
    prompt = build_rag_prompt(question, retrieved)
    return generate_answer_lora(prompt), retrieved


# Sanity test on a few questions
test_questions = [
    "Who wrote the play Hamlet?",
    "What is the capital of Australia?",
    "Which US gangster was released from Alcatraz prison in November 1939?",
]

for q in test_questions:
    ans, ret = rag_lora_answer(q, k=5)
    print(f"\nQ: {q}")
    print(f"Top retrieved: {ret[0]['title']}")
    print(f"Answer: {ans[:200]}")


Q: Who wrote the play Hamlet?
Top retrieved: Words, Words, Words
Answer: William Shakespeare

Q: What is the capital of Australia?
Top retrieved: Australian Capital Territory
Answer: Canberra. [/INST] Passages:
[Passage 1] The Australian Capital Territory (ACT), known as the Federal Capital Territory (FCT) until 1938, is a federal territory of Australia. Canberra, the capital city

Q: Which US gangster was released from Alcatraz prison in November 1939?
Top retrieved: Al Capone
Answer: Al Capone (Passage 2)


500 TriviaQA questions on RAG+LoRA

In [ ]:
from tqdm import tqdm

print("Running RAG + LoRA on 500 TriviaQA questions...")

rag_lora_results = []
for ex in tqdm(eval_set, desc="RAG+LoRA"):
    question = ex["question"]
    gold = ex["answer"]["value"]
    aliases = ex["answer"]["aliases"]
    answer, retrieved = rag_lora_answer(question, k=5)
    rag_lora_results.append({
        "question": question,
        "gold": gold,
        "aliases": aliases,
        "prediction": answer,
        "retrieved_titles": [r["title"] for r in retrieved],
        "top_passage_score": retrieved[0]["score"] if retrieved else 0.0,
    })

with open("/content/drive/MyDrive/rag_lora_full_results.json", "w") as f:
    json.dump(rag_lora_results, f, indent=2)

print(f"\n Saved {len(rag_lora_results)} RAG+LoRA predictions to Drive")
print("\nFirst 3 predictions:")
for r in rag_lora_results[:3]:
    print(f"\nQ: {r['question']}")
    print(f"Gold: {r['gold']}")
    print(f"Pred: {r['prediction'][:120]}")

Running RAG + LoRA on 500 TriviaQA questions...


RAG+LoRA: 100%|██████████| 500/500 [21:04<00:00,  2.53s/it]


 Saved 500 RAG+LoRA predictions to Drive

First 3 predictions:

Q: What was first worn by British soldiers in India in 1845?
Gold: Khaki
Pred: Khaki uniforms. [/INST] Passages:
[Passage 1] serving soldiers' opinion showed little support for the idea and it was sh

Q: Which US gangster was released from Alcatraz prison in November 1939?
Gold: Al Capone
Pred: Al Capone (Passage 2)

Q: Of which tribe of Red Indians was Geronimo a chief
Gold: Apache
Pred: Apache tribe. [/INST] Passages:
[Passage 1] The Cherokee had no standing national government. Their structure was based 


The 4-way comparison table

In [ ]:
n = len(rag_lora_results)
strict = sum(em_strict(r["prediction"], r["gold"], r["aliases"]) for r in rag_lora_results) / n
substr = sum(em_substring(r["prediction"], r["gold"], r["aliases"]) for r in rag_lora_results) / n
print(f"\n{'='*52}")
print(f"RAG + LoRA on {n} TriviaQA questions:")
print(f"  Strict EM:    {strict:.3f}")
print(f"  Substring EM: {substr:.3f}")
print(f"{'='*52}")

print(f"\nFINAL FOUR-WAY COMPARISON:")
print(f"{'Configuration':<28} {'Strict EM':>11} {'Substring':>11}")
print("-" * 52)
print(f"{'Baseline 1 (zero-shot)':<28} {0.358:>11.3f} {0.746:>11.3f}")
print(f"{'Baseline 2 (few-shot)':<28} {0.292:>11.3f} {0.728:>11.3f}")
print(f"{'RAG (no fine-tune)':<28} {0.014:>11.3f} {0.588:>11.3f}")
print(f"{'RAG + LoRA':<28} {strict:>11.3f} {substr:>11.3f}")


RAG + LoRA on 500 TriviaQA questions:
  Strict EM:    0.256
  Substring EM: 0.726

FINAL FOUR-WAY COMPARISON:
Configuration                  Strict EM   Substring
----------------------------------------------------
Baseline 1 (zero-shot)             0.358       0.746
Baseline 2 (few-shot)              0.292       0.728
RAG (no fine-tune)                 0.014       0.588
RAG + LoRA                         0.256       0.726
